## Data Loading

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco2017/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

Using device: cpu


In [3]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset

In [4]:
USE_SUBSET_DATA = False 
train_dataset = load_aokvqa(aokvqa_dir, 'train')  
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")

Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702


## Data Preparation

Fields Considered:

- Question
- Choices
- Correct answer
- Correct Choice Indice
- Rationale
- Direct answer

In [5]:
qa_data = []
for sample in val_dataset:
    question_id = sample["question_id"]
    image_id = sample["image_id"]
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question_id": question_id,
        "image_id": image_id,
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)

In [6]:
qa_df.head()


,question_id,image_id,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer
0,22jbM6gDxdaMaunuzgrsBB,461751,What is in the motorcyclist's mouth?,"[toothpick, food, popsicle stick, cigarette]",cigarette,3,He's smoking while riding. The motorcyclist ha...,cigarette cigarette cigarette cigarette cigare...
1,2Aq5RiEn7eyfWjEbpuYT2o,377368,Which number birthday is probably being celebr...,"[one, ten, nine, thirty]",thirty,3,There is a birthday cake on the table with the...,thirty 30th thirty thirty thirty 30th thirty t...
2,2Br4bJfKY7SQM9DECrqqeG,563603,What best describes the pool of water?,"[frozen, fresh, dirty, boiling]",dirty,2,The pool is dark brown. It it brown and surrou...,muddy dirty murky water muddy pond pond wateri...
3,2C8riXpRLX3CyM5jDz23m7,329542,What is the white substance on top of the cupc...,"[butter, mayo, ice cream, icing]",icing,3,This is frosting used to decorate and add more...,icing whipped cream icing frosting icing frost...
4,2DQex53EkNGH2cfo3WPuPn,182202,What type of device is sitting next to the lap...,"[mouse, mobile phone, pen, keyboard]",mobile phone,1,It has the name of it on the top The device ha...,cell phone vodafone phone phone phone phone ce...


## Multimodal

### CLIP

In [19]:
def get_clip_prediction(img_path, choices, question):
    # Load and preprocess the image
    image = Image.open(img_path).convert("RGB")
    image_input = preprocess(image).unsqueeze(0).to(device)
    
    # Tokenize the list of candidate text choices
    # text_input = clip.tokenize(choices).to(device)
    text_input = torch.cat([
        clip.tokenize(f"{question} Answer: {c}") for c in choices
    ]).to(device)
    # print(str(text_input))
    
    with torch.no_grad():
        # Compute image and text features
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)
        
        # Normalize features to unit length (recommended for cosine similarity)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity between the image and each text choice
        similarity = (image_features @ text_features.T).squeeze(0)

        confidence_scores = torch.softmax(similarity, dim=0).cpu().numpy()
    
    # Return the choice with the highest similarity score
    best_idx = similarity.argmax().item()
    best_answer = choices[best_idx]
    best_confidence = confidence_scores[best_idx]
    
    return best_answer, best_confidence

In [20]:
import torch
import clip
from PIL import Image
from tqdm import tqdm

# Set up device and load the CLIP model along with its preprocessing pipeline.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

# Process each sample in qa_data and compute predictions
predictions = []
condifence_scores = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction, confidence = get_clip_prediction(img_path, choices, sample['question'])
    predictions.append(prediction)
    condifence_scores.append(confidence)

qa_df['clip_baseline_prediction'] = predictions
qa_df['clip_baseline_confidence'] = condifence_scores


Processing Images: 100%|██████████| 1145/1145 [06:44<00:00,  2.83it/s]


In [23]:
correctness = (qa_df['clip_baseline_prediction'] == qa_df['correct_answer'])
accuracy = correctness.mean()
calibration_error = (qa_df['clip_baseline_confidence'] - correctness).abs().mean()
print(f"CLIP Baseline Accuracy: {accuracy:.2%}")
print(f"CLIP Baseline Calibration Error: {calibration_error:.2%}")

CLIP Baseline Accuracy: 55.11%
CLIP Baseline Calibration Error: 52.41%
